# Marswalk — AI4Mars terrain segmentation (REAL DATA)

Trains a **semantic segmentation** model on the real **AI4Mars** dataset (Swan,
Ono, Atha et al., NASA/JPL) to label rover-view terrain into
**soil / bedrock / sand / big-rock**. This is a *different task* from the
boulder detector (`train_rock_detector.ipynb`): per-pixel terrain classes on
rover images, not boxes on orbital images. The **big-rock** class is the
routing-relevant hazard output.

| | |
|---|---|
| Dataset | AI4Mars merged v0.6 — [doi:10.5281/zenodo.15995036](https://doi.org/10.5281/zenodo.15995036) |
| License | **CC BY 4.0** (Zenodo record) |
| Size | `ai4mars-dataset-merged-0.6.zip` ≈ **16.2 GB** (md5 `daf80a86021253292e6c425f97baa5c6`) |
| Classes (NAV key, verified from `label_keys.json`) | `0` soil · `1` bedrock · `2` sand · `3` big rock · `255` NULL/ignore |
| Split | `labels/train/` (crowd, ≥3 labelers, 2/3 pixel agreement) · `labels/test/masked-gold-min{1,2,3}-100agree/` (expert) |
| Subset used | **MSL NavCam** (`msl/ncam`) — the cleanest subset; rover + 30 m range already masked to 255 |

**Attribution (CC BY 4.0):** cite AI4Mars — Swan, R. M., Atha, D., Ono, M., et al.,
*AI4MARS: A Dataset for Terrain-Aware Autonomous Driving on Mars*, CVPRW 2021
(doi:10.1109/CVPRW53098.2021.00226) / IEEE Aerospace 2022
(doi:10.1109/AERO53065.2022.9843615). Data: doi:10.5281/zenodo.15995036.

### Honest status — read before trusting any number
- **Run on the owner machine (RTX 4050 + the 16 GB download).** The Marswalk
  sandbox does not download 16 GB; the loader/model/metric *code* was verified on
  **real extracted MSL NavCam patches**, but the *trained result is owner-side*.
- `M2020_GEO` labels use a **different** geology key (values by tens) — this
  notebook uses the 4-class **NAV** key only and skips `M2020_GEO`.
- **Gate (MARSWALK_CONTEXT.md §3):** terrain/rock classes inform routing only
  after the P6 ablation shows they change route decisions vs slope-only.
- AI4Mars is **rover-view** (navcam), not map-registered orbital data — it does
  not directly give a top-down hazard raster; it classifies what the rover sees.
  For the orbital boulder hazard layer use `train_rock_detector.ipynb`.

## 0. Setup

In [ ]:
import sys, pathlib, time, json, hashlib, urllib.request, zipfile
import numpy as np
import torch, torchvision

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch", torch.__version__, "| torchvision", torchvision.__version__, "| device", device)
if device.type != "cuda":
    print("WARNING: no CUDA — training will be slow; use the RTX 4050.")

## 1. Config (real Zenodo file; classes verified from `label_keys.json`)

In [ ]:
AI4MARS = {
    "record": "15995036",
    "zip": "ai4mars-dataset-merged-0.6.zip",
    "url": "https://zenodo.org/api/records/15995036/files/ai4mars-dataset-merged-0.6.zip/content",
    "md5": "daf80a86021253292e6c425f97baa5c6", "size": 16_232_481_989,
}
CLASS_NAMES = ["soil", "bedrock", "sand", "big rock"]  # NAV pixel values 0..3
NUM_CLASSES = 4
IGNORE = 255           # NULL / rover / >30 m range (already baked into ncam labels)
SUBSET = "msl/ncam"    # cleanest subset; also 'mer', 'msl/mcam', etc.
TEST_VERSION = "min1"  # expert test gold: min1 | min2 | min3 (min3 strictest/sparsest)
IMG_SIZE = 512         # AI4Mars images are 1024x1024; downsize for the 6 GB GPU
DATA_DIR = pathlib.Path("data/ai4mars")   # gitignored; put the download here
print("download target:", DATA_DIR.resolve())

## 2. Download (owner step — ~16 GB)

One MD5-verified zip. This is large; it is the real dataset. Re-running skips a
file already present with a matching checksum.

In [ ]:
GET = True

def _md5(path, chunk=1 << 20):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()

def download(spec, dest_dir):
    dest_dir.mkdir(parents=True, exist_ok=True)
    out = dest_dir / spec["zip"]
    if out.exists() and _md5(out) == spec["md5"]:
        print("ok (cached):", spec["zip"]); return out
    print(f"downloading {spec['zip']} (~{spec['size']/1e9:.1f} GB) ...")
    with urllib.request.urlopen(spec["url"]) as r, open(out, "wb") as f:
        done = 0
        while True:
            buf = r.read(1 << 20)
            if not buf:
                break
            f.write(buf); done += len(buf)
            if done % (500 << 20) < (1 << 20):
                print(f"  {done/1e9:5.1f}/{spec['size']/1e9:.1f} GB", end="\r")
    if _md5(out) != spec["md5"]:
        raise RuntimeError(f"MD5 mismatch for {spec['zip']}")
    print("\nverified", spec["zip"]); return out

if GET:
    zpath = download(AI4MARS, DATA_DIR)
    with zipfile.ZipFile(zpath) as z:
        z.extractall(DATA_DIR)
    print("extracted to", DATA_DIR)

## 3. Load real (image, label) pairs — MSL NavCam, NAV 4-class key

In [ ]:
def _subset_base(data_root, subset):
    # locate the subset dir (…/msl/ncam) regardless of the version root folder
    want = subset.replace("\\", "/")
    for p in pathlib.Path(data_root).rglob("train"):
        s = str(p).replace("\\", "/")
        if s.endswith("/labels/train") and want in s:
            return p.parent.parent  # …/msl/ncam
    raise FileNotFoundError(f"could not find {subset}/labels/train under {data_root}")

def _img_for(edr_dir, stem):
    for ext in (".JPG", ".jpg", ".jpeg"):
        p = edr_dir / f"{stem}{ext}"
        if p.exists():
            return p
    return None

def load_pairs(data_root, subset=SUBSET, test_version=TEST_VERSION):
    base = _subset_base(data_root, subset)
    edr = base / "images" / "edr"
    train = []
    for lbl in (base / "labels" / "train").glob("*.png"):
        img = _img_for(edr, lbl.stem)                  # train label stem == image stem
        if img:
            train.append((img, lbl))
    tdir = base / "labels" / "test" / f"masked-gold-{test_version}-100agree"
    test = []
    if tdir.exists():
        for lbl in tdir.glob("*.png"):
            img = _img_for(edr, lbl.stem.replace("_merged", ""))  # test stem has _merged
            if img:
                test.append((img, lbl))
    print(f"subset {base.as_posix().split('/')[-2]}/{base.name} | train {len(train)} / "
          f"test {len(test)} (gold {test_version})")
    if not train:
        raise RuntimeError("no train pairs — check SUBSET and that the zip extracted.")
    return train, test

train_pairs, test_pairs = load_pairs(DATA_DIR)

## 4. Dataset + model (torchvision DeepLabV3-ResNet50, 4 classes)

In [ ]:
from torch.utils.data import DataLoader, Dataset
import torch.nn.functional as F

def _load_img(path, size):
    from PIL import Image
    a = np.asarray(Image.open(path).convert("L"), dtype=np.float32) / 255.0
    t = torch.from_numpy(a)[None, None]                         # 1x1xHxW
    t = F.interpolate(t, size=(size, size), mode="bilinear", align_corners=False)
    return t[0].repeat(3, 1, 1)                                 # 3xHxW

def _load_label(path, size):
    from PIL import Image
    a = np.asarray(Image.open(path), dtype=np.int64)
    if a.ndim == 3:
        a = a[..., 0]
    a[(a < 0) | (a >= NUM_CLASSES)] = IGNORE                    # keep 0..3; everything else ignore
    t = torch.from_numpy(a)[None, None].float()
    t = F.interpolate(t, size=(size, size), mode="nearest")     # nearest: preserve class ids
    return t[0, 0].long()

class AI4MarsDataset(Dataset):
    def __init__(self, pairs, size=IMG_SIZE):
        self.pairs, self.size = pairs, size
    def __len__(self):
        return len(self.pairs)
    def __getitem__(self, i):
        img_p, lbl_p = self.pairs[i]
        return _load_img(img_p, self.size), _load_label(lbl_p, self.size)

def build_model(num_classes=NUM_CLASSES, pretrained=True):
    w = "DEFAULT" if pretrained else None
    m = torchvision.models.segmentation.deeplabv3_resnet50(weights=None, weights_backbone=w)
    m.classifier[-1] = torch.nn.Conv2d(256, num_classes, 1)     # 21 COCO classes -> 4
    m.aux_classifier = None
    return m

model = build_model().to(device)
print("model ready:", sum(p.numel() for p in model.parameters()) / 1e6, "M params")

## 5. Train — CrossEntropy(ignore=255), AMP + channels_last + grad accum

Logs **peak VRAM** and **epoch time**. On 6 GB keep `IMG_SIZE`=512, `BATCH`=2–4.

In [ ]:
EPOCHS, BATCH, GRAD_ACCUM, LR = 20, 4, 1, 1e-3
AMP = True; CHANNELS_LAST = True

def train(model, pairs):
    # drop_last=True: DeepLabV3's ASPP pooling + BatchNorm needs batch > 1
    dl = DataLoader(AI4MarsDataset(pairs), batch_size=BATCH, shuffle=True,
                    num_workers=0, drop_last=len(pairs) >= BATCH)
    use_amp = AMP and device.type == "cuda"; cl = CHANNELS_LAST and device.type == "cuda"
    if cl:
        model = model.to(memory_format=torch.channels_last)
    opt = torch.optim.AdamW(model.parameters(), lr=LR)
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    lossfn = torch.nn.CrossEntropyLoss(ignore_index=IGNORE)
    model.train()
    for ep in range(EPOCHS):
        if device.type == "cuda":
            torch.cuda.reset_peak_memory_stats()
        t0, run = time.time(), 0.0; opt.zero_grad()
        for step, (x, y) in enumerate(dl):
            x = x.to(device, memory_format=torch.channels_last) if cl else x.to(device)
            y = y.to(device)
            with torch.autocast(device_type=device.type, enabled=use_amp):
                logits = model(x)["out"]
                loss = lossfn(logits, y) / GRAD_ACCUM
            scaler.scale(loss).backward()
            if (step + 1) % GRAD_ACCUM == 0:
                scaler.step(opt); scaler.update(); opt.zero_grad()
            run += float(loss) * GRAD_ACCUM
        peak = (torch.cuda.max_memory_allocated() / 1e9) if device.type == "cuda" else float("nan")
        print(f"epoch {ep+1}/{EPOCHS} | loss {run/max(len(dl),1):.3f} | {time.time()-t0:.1f}s | peak VRAM {peak:.2f} GB")
    return model

# Tip: set EPOCHS=1 first to confirm the loop + VRAM fit, then raise it.
model = train(model, train_pairs)

## 6. Evaluate — per-class IoU + mIoU (big-rock IoU is the hazard metric)

In [ ]:
@torch.no_grad()
def evaluate(model, pairs):
    model.eval()
    conf = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=np.int64)
    dl = DataLoader(AI4MarsDataset(pairs), batch_size=BATCH, num_workers=0)
    for x, y in dl:
        pred = model(x.to(device))["out"].argmax(1).cpu().numpy().ravel()
        tgt = y.numpy().ravel()
        m = tgt != IGNORE
        k = tgt[m] * NUM_CLASSES + pred[m]
        conf += np.bincount(k, minlength=NUM_CLASSES**2).reshape(NUM_CLASSES, NUM_CLASSES)
    ious = []
    print(f"{'class':>10} | {'IoU':>6} | pixels")
    for c in range(NUM_CLASSES):
        tp = conf[c, c]; fn = conf[c].sum() - tp; fp = conf[:, c].sum() - tp
        iou = tp / (tp + fp + fn) if (tp + fp + fn) else float("nan")
        ious.append(iou)
        print(f"{CLASS_NAMES[c]:>10} | {iou:6.3f} | {conf[c].sum()}")
    miou = np.nanmean(ious)
    print(f"\nmIoU: {miou:.3f} | big-rock IoU (hazard): {ious[3]:.3f}")
    return ious

if test_pairs:
    _ = evaluate(model, test_pairs)
else:
    print("no expert test set for this subset/version; evaluate on a held-out train split.")

## 7. Format self-check (no download needed)

Runs the dataset→model→loss→IoU path on a tiny in-memory sample in the real
AI4Mars format (grayscale image + label with values {0,1,2,3,255}). This is a
**code check, not training data**.

In [ ]:
def _selfcheck():
    import tempfile, os
    from PIL import Image
    d = pathlib.Path(tempfile.mkdtemp())
    (d / "images" / "edr").mkdir(parents=True); (d / "labels" / "train").mkdir(parents=True)
    img = (np.random.rand(64, 64) * 255).astype(np.uint8)
    lbl = np.full((64, 64), IGNORE, np.uint8); lbl[:32, :32] = 3; lbl[32:, :] = 0  # big rock + soil
    Image.fromarray(img).save(d / "images" / "edr" / "X.JPG")
    Image.fromarray(lbl).save(d / "labels" / "train" / "X.png")
    ds = AI4MarsDataset([(d / "images" / "edr" / "X.JPG", d / "labels" / "train" / "X.png")], size=64)
    x, y = ds[0]
    assert x.shape == (3, 64, 64) and y.shape == (64, 64)
    assert set(np.unique(y.numpy()).tolist()) <= {0, 3, IGNORE}
    m = build_model(pretrained=False).eval()  # eval: BatchNorm ok on a batch of 1
    out = m(x[None])["out"]
    assert out.shape[1] == NUM_CLASSES
    loss = torch.nn.CrossEntropyLoss(ignore_index=IGNORE)(out, y[None])
    assert torch.isfinite(loss)
    print("format self-check OK — loss", round(float(loss), 3))

_selfcheck()

## 8. What to paste back to the team

After a real run on the RTX 4050:
- [ ] **Peak VRAM** (GB) and **epoch time** (s) at the `IMG_SIZE`/`BATCH` used.
- [ ] **per-class IoU + mIoU** on the expert gold test, especially **big-rock IoU**.
- [ ] Subset (`msl/ncam` …), test gold version (`min1/2/3`), train/test counts.
- [ ] Any config changes from defaults.

Then update `AI_LOG.md` (P2) and `DATA_LICENSES.md` (AI4Mars = CC BY 4.0 per
Zenodo; reconcile with the NASA-portal note). **Cite Swan/Atha/Ono et al. (CVPRW
2021 / IEEE Aero 2022).** Terrain classes inform routing only after the P6
ablation; big-rock is the hazard-relevant class.